In [ ]:
import pandas as pd

#### Cleanse HKHS Catalogue

In [ ]:
cat = pd.read_parquet("../data/raw/categories.parquet")
cat.columns = [
    "lb", "ub", "id", "label",
    "chapter", "cat_en", "cat_zht", "seq",
    "cat_zhs",
]

cat["hs"] = "HS2"
cat["hsc"] = [
    [str(i).zfill(2) for i in range(int(lb), int(ub) + 1)]
    for lb, ub in zip(cat["lb"], cat["ub"])
]
cat = cat.explode("hsc", ignore_index=True)
cat.query("hsc != '77'", inplace=True)  # exclude 77 as it is not used in HKHS

cat.rename(columns={"cat_en": "desc"}, inplace=True)
cat = cat[["hs", "hsc", "desc"]]

assert len(cat["hsc"]) == 97

In [ ]:
subcat = pd.read_parquet("../data/raw/subcategories.parquet")
subcat.columns = [
    "hsc", "subcat_en", "subcat_zht", "subcat_zhs",
    "uqc", "unit_desc",
]

mask = subcat["hsc"].str.len().eq(8)
subcat.loc[mask, "hsc"] = subcat.loc[mask, "hsc"].str[:6]
subcat.drop_duplicates(subset=["hsc"], inplace=True)

subcat["hs"] = "HS" + subcat["hsc"].str.len().astype(str)
subcat["hs_fk"] = subcat["hsc"].str[:-2]

left = subcat.query("hs_fk.str.len() != 0")[["hs", "hsc", "hs_fk"]]
right = subcat[["hsc", "subcat_en"]]
subcat = left.merge(
    right,
    left_on="hs_fk",
    right_on="hsc",
    suffixes=("", "_parent"),
)

subcat.rename(columns={"subcat_en": "desc"}, inplace=True)
subcat = subcat[["hs", "hsc", "desc"]]

assert len(subcat["hsc"]) == 1_229 + 5_613

#### Cleanse HKHS Keyword

In [ ]:
desc = pd.read_csv(
    "../data/raw/B2XX00232026XXXXB0100.csv",
    dtype=str,
    skiprows=1,
)
desc.columns = ["hkhs", "desc_zh", "desc_en", "uqc", "unit_chi", "unit_eng"]

desc["hkhs"] = desc["hkhs"].str.zfill(8)
desc[["chapter", "heading", "subheading"]] = pd.concat(
    [desc["hkhs"].str[:n] for n in (2, 4, 6)],
    axis=1,
).to_numpy()
desc["desc_en"] = desc["desc_en"].str.replace(r"\s+", " ", regex=True) \
    .str.strip() \
    .str.lower()
desc["field"] = "description"

desc.rename(columns={"desc_en": "desc"}, inplace=True)
desc = desc[["chapter", "heading", "subheading", "hkhs", "desc", "field"]]

assert desc["chapter"].nunique() == 97
assert desc["heading"].nunique() == 1_229
assert desc["subheading"].nunique() == 5_613
assert len(desc) == 7_822

In [ ]:
cmdty = pd.read_parquet("../data/raw/keywords.parquet")
cmdty.columns = ["hkhs", "kwd_en", "kwd_zh", "kwds"]

cmdty[["chapter", "heading", "subheading"]] = pd.concat(
    [cmdty["hkhs"].str[:n] for n in (2, 4, 6)],
    axis=1,
).to_numpy()
cmdty["kwd_en"] = cmdty["kwd_en"].str.replace(r"\s+", " ", regex=True) \
    .str.strip() \
    .str.lower()
cmdty["field"] = "commodity"

cmdty.drop_duplicates(subset=["subheading", "kwd_en"], inplace=True)
cmdty.rename(columns={"kwd_en": "desc"}, inplace=True)
cmdty = cmdty[["chapter", "heading", "subheading", "hkhs", "desc", "field"]]

assert cmdty["chapter"].nunique() == 97
assert cmdty["heading"].nunique() == 1_229
assert cmdty["subheading"].nunique() == 5_613

In [ ]:
hsc = pd.read_html("../data/raw/index.php", header=0)[0]
hsc.columns = ["kana", "japanese", "chinese", "english", "tariff_sched"]

hsc["tariff_sched"] = hsc["tariff_sched"].str.replace("HSコード類の付定例", "")
hsc[["chapter", "heading", "subheading"]] = pd.concat(
    [hsc["tariff_sched"].str[:n] for n in (2, 4, 6)],
    axis=1,
).to_numpy()
hsc["hkhs"] = hsc["subheading"] + "00"
hsc.query("chapter in @desc.chapter.unique()", inplace=True)

hsc["english"] = hsc["english"].str.replace(r"\s+", " ", regex=True) \
    .str.strip() \
    .str.lower()
for pattern in ["英語", "【not acceptable】", "【not acceptable in general】"]:
    hsc["english"] = hsc["english"].str.replace(pattern, "", case=False)
hsc["field"] = "example"

hsc.drop_duplicates(subset=["subheading", "english"], inplace=True)
hsc.rename(columns={"english": "desc"}, inplace=True)
hsc = hsc[["chapter", "heading", "subheading", "hkhs", "desc", "field"]]

#### Normalise Data

In [ ]:
nest = desc[["hkhs", "desc"]].assign(hs="HS8").rename(columns={"hkhs": "hsc"})
catalogue = pd.concat([cat, subcat, nest], ignore_index=True)
catalogue["desc"] = catalogue["desc"].str.capitalize()
catalogue.sort_values(by=["hs", "hsc"], inplace=True)

In [ ]:
hkhs = pd.concat([desc, cmdty, hsc], ignore_index=True)
hkhs.sort_values(by=["field", "hkhs"], inplace=True)

In [ ]:
catalogue.to_parquet("../data/processed/catalogue.parquet", index=False)
hkhs.to_parquet("../data/interim/hkhs.parquet", index=False)